In [ ]:
# =========================================================
# 0. INSTALL LIBRARIES & SETUP DEVICE
# =========================================================
!pip install -U transformers datasets soundfile librosa torchcodec

import os
import torch
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
import seaborn as sns

from datasets import Dataset, Audio
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
)
from transformers import (
    Wav2Vec2Processor,
    Wav2Vec2ForSequenceClassification,
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    TrainingArguments,
    Trainer,
)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# Disable wandb completely
os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"


# # =========================================================
# # 1. TEXT BRANCH – TRAIN DISTILBERT ON ENRON EMAIL FATIGUE
# # =========================================================
# # 🔴 TODO: UPDATE THESE PATHS TO YOUR ENRON TRAIN/TEST CSV FILES
ENRON_TRAIN_CSV = "/content/drive/MyDrive/Colab Notebooks/fatigue detection/data/enron_train.csv"
ENRON_TEST_CSV  = "/content/drive/MyDrive/Colab Notebooks/fatigue detection/data/enron_test.csv"

# Assumed columns: 'clean_text' (email text), 'label' (class name: Fatigue/Neutral/Stress)
df_train = pd.read_csv(ENRON_TRAIN_CSV)
df_test  = pd.read_csv(ENRON_TEST_CSV)

print("Train shape:", df_train.shape)
print("Test shape:", df_test.shape)
print("Train label distribution:\n", df_train["label"].value_counts())

# Build label mappings from train set
text_label_names = sorted(df_train["label"].unique())
text_label2id = {lab: i for i, lab in enumerate(text_label_names)}
text_id2label = {i: lab for lab, i in text_label2id.items()}
print("\nText Label2ID:", text_label2id)

df_train["label_id"] = df_train["label"].map(text_label2id)
df_test["label_id"]  = df_test["label"].map(text_label2id)

# Create HF datasets
# 🔴 If your text column is not 'clean_text', change it here.
train_text_ds = Dataset.from_pandas(df_train[["clean_text", "label_id"]])
test_text_ds  = Dataset.from_pandas(df_test[["clean_text", "label_id"]])

text_tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

def tokenize_text(batch):
    return text_tokenizer(
        batch["clean_text"],
        padding="max_length",
        truncation=True,
        max_length=256,
    )

train_text_ds = train_text_ds.map(tokenize_text, batched=True)
test_text_ds  = test_text_ds.map(tokenize_text, batched=True)

# Remove raw text column and rename label
train_text_ds = train_text_ds.remove_columns(["clean_text"])
test_text_ds  = test_text_ds.remove_columns(["clean_text"])

train_text_ds = train_text_ds.rename_column("label_id", "labels")
test_text_ds  = test_text_ds.rename_column("label_id", "labels")

train_text_ds.set_format(type="torch")
test_text_ds.set_format(type="torch")

# Load DistilBERT model
text_model = DistilBertForSequenceClassification.from_pretrained(
     "distilbert-base-uncased",
     num_labels=len(text_label_names),
     id2label=text_id2label,
     label2id=text_label2id,
).to(DEVICE)

def text_compute_metrics(pred):
    y_pred = np.argmax(pred.predictions, axis=-1)
    y_true = pred.label_ids
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    }

text_training_args = TrainingArguments(
    output_dir="./distilbert_enron_fatigue",
    learning_rate=1e-5,
    num_train_epochs=20,            # increase if needed
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    do_eval=True,
    report_to="none",
)

text_trainer = Trainer(
    model=text_model,
    args=text_training_args,
    train_dataset=train_text_ds,
    eval_dataset=test_text_ds,
    tokenizer=text_tokenizer,
    compute_metrics=text_compute_metrics,
)

print("\n=== Training DistilBERT on Enron emails ===")
text_trainer.train()

print("\n=== Evaluating DistilBERT on test emails ===")
text_preds = text_trainer.predict(test_text_ds)
y_true_text = text_preds.label_ids
y_pred_text = np.argmax(text_preds.predictions, axis=1)
print("\nClassification Report (text only):")
print(classification_report(y_true_text, y_pred_text, target_names=text_label_names))

# Save text model for later inference
TEXT_MODEL_DIR = "./models/distilbert_fatigue_model"
os.makedirs(TEXT_MODEL_DIR, exist_ok=True)
text_trainer.save_model(TEXT_MODEL_DIR)
text_tokenizer.save_pretrained(TEXT_MODEL_DIR)
print("\nSaved text model to:", TEXT_MODEL_DIR)


# =========================================================
# 2. AUDIO BRANCH – TRAIN WAV2VEC2 ON RAVDESS (FATIGUE/STRESS/NEUTRAL)
# =========================================================
RAVDESS_ZIP_URL = "https://zenodo.org/records/1188976/files/Audio_Speech_Actors_01-24.zip?download=1"

if not os.path.exists("ravdess_speech.zip"):
    !wget -O ravdess_speech.zip "$RAVDESS_ZIP_URL"

if not os.path.exists("ravdess_speech"):
    !unzip -q ravdess_speech.zip -d ravdess_speech/

print("\nFolders in ravdess_speech:")
!ls ravdess_speech

BASE_DIR = "ravdess_speech"  # contains Actor_01 ... Actor_24

emotion_code_to_name = {
    1: "neutral",
    2: "calm",
    3: "happy",
    4: "sad",
    5: "angry",
    6: "fearful",
    7: "disgust",
    8: "surprised",
}

# Map basic emotions -> your fatigue classes
emo_to_target = {
    "sad": "Fatigue",
    "angry": "Stress",
    "fearful": "Stress",
    "disgust": "Stress",
    "neutral": "Neutral",
    "calm": "Neutral",
    "happy": "Neutral",
    "surprised": "Neutral",
}

rows = []
for actor_folder in sorted(os.listdir(BASE_DIR)):
    actor_path = os.path.join(BASE_DIR, actor_folder)
    if not os.path.isdir(actor_path):
        continue

    for fname in os.listdir(actor_path):
        if not fname.lower().endswith(".wav"):
            continue

        fpath = os.path.join(actor_path, fname)
        parts = fname.split(".")[0].split("-")  # MM-VC-EE-II-SS-RR-AA
        if len(parts) != 7:
            continue

        emo_code = int(parts[2])  # EE
        emo_name = emotion_code_to_name.get(emo_code)
        target = emo_to_target.get(emo_name)
        if target is None:
            continue

        rows.append({
            "path": fpath,
            "emotion": emo_name,
            "target_label": target,
        })

df_audio = pd.DataFrame(rows)
print("\nAudio manifest head:")
print(df_audio.head())

print("\nOriginal emotion distribution:")
print(df_audio["emotion"].value_counts())

print("\nTarget label distribution:")
print(df_audio["target_label"].value_counts())

df_audio.to_csv("ravdess_fatigue_manifest.csv", index=False)


# =========================================================
# 3. BUILD HF AUDIO DATASET + TRAIN WAV2VEC2
# =========================================================
df_audio = pd.read_csv("ravdess_fatigue_manifest.csv")

target_labels = sorted(df_audio["target_label"].unique())  # e.g. ['Fatigue','Neutral','Stress']
label2id = {lab: i for i, lab in enumerate(target_labels)}
id2label = {i: lab for lab, i in label2id.items()}
print("\nAudio Label2ID mapping:", label2id)

df_audio["label_id"] = df_audio["target_label"].map(label2id)

hf_audio_ds = Dataset.from_pandas(df_audio[["path", "label_id"]])
hf_audio_ds = hf_audio_ds.cast_column("path", Audio(sampling_rate=16000))

dataset_audio = hf_audio_ds.train_test_split(test_size=0.2, seed=42)
train_audio_ds = dataset_audio["train"]
test_audio_ds  = dataset_audio["test"]

print("\nAudio Train / Test sizes:", len(train_audio_ds), len(test_audio_ds))

audio_processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

audio_model = Wav2Vec2ForSequenceClassification.from_pretrained(
    "facebook/wav2vec2-base",
    num_labels=len(target_labels),
    label2id=label2id,
    id2label=id2label,
).to(DEVICE)

MAX_LEN_SAMPLES = 5 * 16000  # 5 seconds at 16kHz

def preprocess_audio(batch):
    audio = batch["path"]["array"]
    sr = batch["path"]["sampling_rate"]
    inputs = audio_processor(
        audio,
        sampling_rate=sr,
        return_tensors="pt",
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN_SAMPLES,
    )
    batch["input_values"] = inputs["input_values"][0]
    return batch

train_audio_proc = train_audio_ds.map(preprocess_audio)
test_audio_proc  = test_audio_ds.map(preprocess_audio)

train_audio_proc = train_audio_proc.remove_columns(["path"])
test_audio_proc  = test_audio_proc.remove_columns(["path"])

train_audio_proc = train_audio_proc.rename_column("label_id", "labels")
test_audio_proc  = test_audio_proc.rename_column("label_id", "labels")

train_audio_proc.set_format(type="torch")
test_audio_proc.set_format(type="torch")

print("\nOne processed audio example:")
print(train_audio_proc[0])

def audio_compute_metrics(pred):
    y_pred = np.argmax(pred.predictions, axis=-1)
    y_true = pred.label_ids
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_weighted": f1_score(y_true, y_pred, average="weighted"),
    }

audio_training_args = TrainingArguments(
    output_dir="./wav2vec_ravdess_fatigue",
    learning_rate=1e-5,
    num_train_epochs=20,             # increase if needed
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    do_eval=True,
    report_to="none",
)

audio_trainer = Trainer(
    model=audio_model,
    args=audio_training_args,
    train_dataset=train_audio_proc,
    eval_dataset=test_audio_proc,
    tokenizer=audio_processor,
    compute_metrics=audio_compute_metrics,
)

print("\n=== Training Wav2Vec2 on RAVDESS audio ===")
audio_trainer.train()

print("\n=== Evaluating Wav2Vec2 on test audio ===")
audio_preds = audio_trainer.predict(test_audio_proc)
y_true_audio = audio_preds.label_ids
y_pred_audio = np.argmax(audio_preds.predictions, axis=1)
print("\nClassification Report (audio only):")
print(classification_report(y_true_audio, y_pred_audio, target_names=target_labels))

cm = confusion_matrix(y_true_audio, y_pred_audio)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_labels, yticklabels=target_labels)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("RAVDESS Wav2Vec2 – Fatigue / Neutral / Stress")
plt.show()

# Save audio model
AUDIO_MODEL_DIR = "./wav2vec_ravdess_fatigue"
audio_trainer.save_model(AUDIO_MODEL_DIR)
audio_processor.save_pretrained(AUDIO_MODEL_DIR)
print("\nSaved audio model to:", AUDIO_MODEL_DIR)


# =========================================================
# 4. INFERENCE HELPERS FOR TEXT + AUDIO
# =========================================================
# Build mapping from text model labels -> target_labels (for fusion)
def build_text_label_order(target_labels):
    mapping = {}
    for idx, name in text_model.config.id2label.items():
        if name in target_labels:
            mapping[idx] = target_labels.index(name)
    return mapping

text_idx_to_target_idx = build_text_label_order(target_labels)
print("\nText index -> target_labels index mapping:", text_idx_to_target_idx)

def predict_text_probs(email_text: str):
    enc = text_tokenizer(
        email_text,
        truncation=True,
        padding=True,
        max_length=256,
        return_tensors="pt",
    ).to(DEVICE)

    with torch.no_grad():
        logits = text_model(**enc).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

    ordered = np.zeros(len(target_labels), dtype=np.float32)
    for text_idx, target_idx in text_idx_to_target_idx.items():
        ordered[target_idx] = probs[text_idx]
    return ordered

def predict_audio_probs(audio_path: str):
    speech, sr = sf.read(audio_path)

    if speech.ndim > 1:
        speech = np.mean(speech, axis=1)

    if sr != 16000:
        speech = librosa.resample(speech, orig_sr=sr, target_sr=16000)
        sr = 16000

    inputs = audio_processor(
        speech,
        sampling_rate=sr,
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    with torch.no_grad():
        logits = audio_model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    return probs

def predict_fused(email_text: str, audio_path: str, w_text: float = 0.5, w_audio: float = 0.5):
    audio_probs = predict_audio_probs(audio_path)
    text_probs  = predict_text_probs(email_text)

    total_w = w_text + w_audio
    w_text_norm  = w_text / total_w
    w_audio_norm = w_audio / total_w

    fused_probs = w_audio_norm * audio_probs + w_text_norm * text_probs
    fused_idx   = int(np.argmax(fused_probs))
    fused_label = target_labels[fused_idx]

    return {
        "audio_probs": audio_probs,
        "text_probs": text_probs,
        "fused_probs": fused_probs,
        "fused_label": fused_label,
    }


# =========================================================
# 5. DEMO: FULL MULTIMODAL PREDICTION
# =========================================================
# Take a sample email from your Enron test set
example_email = df_test.sample(1).iloc[0]["clean_text"]

# Take a random RAVDESS sample
sample_audio = df_audio.sample(1).iloc[0]
example_audio_path = sample_audio["path"]

print("\n=== MULTIMODAL DEMO ===")
print("Audio file:", example_audio_path)
print("Audio true label:", sample_audio["target_label"])
print("\nEmail text:\n", example_email[:500], "...")


result = predict_fused(example_email, example_audio_path)

print("\nLabels order:", target_labels)
print("Audio probs: ", result["audio_probs"])
print("Text probs:  ", result["text_probs"])
print("Fused probs: ", result["fused_probs"])
print("Fused label: ", result["fused_label"])


Using device: cuda
Train shape: (144304, 6)
Test shape: (36077, 6)
Train label distribution:
 label
Neutral    140137
Stress       3473
Fatigue       496
Apathy        198
Name: count, dtype: int64

Text Label2ID: {'Apathy': 0, 'Fatigue': 1, 'Neutral': 2, 'Stress': 3}


Map:   0%|          | 0/144304 [00:00<?, ? examples/s]

Map:   0%|          | 0/36077 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipython-input-1993586968.py:117: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  text_trainer = Trainer(



=== Training DistilBERT on Enron emails ===


Step,Training Loss
500,0.198900
1000,0.158600
1500,0.141400
2000,0.119500
2500,0.137500
3000,0.107800
3500,0.096500
4000,0.084600
4500,0.074400
5000,0.067700
